In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chicago_housing import config as C
from chicago_housing import constants as K
from chicago_housing.data.load import load_training_data
from chicago_housing.data import clean, distributions
from chicago_housing.features import assemble
from chicago_housing.features.spatial import add_distance_to_loop
from chicago_housing.analysis import sales_descriptives as sd
from chicago_housing.viz import charts, maps

pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')

### 1. Basic Cleaning, Profiling and Distributions

In [ ]:
# Load up training parquet and filter for Chicago, SF from 2022-2025
df = load_training_data(columns=K.analysis_columns())   # read
df = clean.scope_filter(df)
print(f"Chicago sales dataframe has shape: {df.shape}")

In [ ]:
# Step 1 — profile the pruned candidate columns: raw dtypes, missingness, degeneracy.
# (engineered cols aren't in the raw frame yet, so exclude K.ENGINEERED.)
candidate = (
    K.BLOCK_A_STRUCTURE
    + [c for c in K.BLOCK_B_LOCATION if c not in K.ENGINEERED]
    + K.DEMOGRAPHICS
)
clean.profile_columns(df, candidate)

In [ ]:
# Step 2 — fixes discovered from the profile above, recorded in constants.py:
#   CHANGE_DTYPE_FROM_FLOAT_TO_INT — counts/years stored as float -> Int64
#   DROP_REDUNDANT_COLS_WRANGLING  — redundant / superseded columns
print("float -> Int64:", K.CHANGE_DTYPE_FROM_FLOAT_TO_INT)
print("drop redundant:", K.DROP_REDUNDANT_COLS_WRANGLING)

# Re-profile the CLEANED frame to confirm the fixes took: dtypes flip to Int64,
# and any dropped candidate shows 'COLUMN NOT FOUND'.
clean.profile_columns(df, candidate, convert_dtypes=True, drop_columns=True)

In [ ]:
# Step 3 — examine distributions of the cleaned numeric columns: decide transforms,
# spot heavy tails. summarize_distributions() = the decision table (skew, tail_ratio,
# transform_hint); plot_distributions() = the shapes.
df_clean = clean.drop_redundant_columns(clean.convert_float_to_int(df), verbose=False)
summary = distributions.summarize_distributions(df_clean, candidate)
display(summary)

distributions.plot_distributions(df_clean, [c for c in candidate if c in df_clean.columns])
plt.show()

## 3. Column profile — missingness + degeneracy
A column can be 0% missing yet useless (one value in >95% of rows). `pct_modal` catches that.

In [ ]:
candidate = (
    C.BLOCK_A_STRUCTURE
    + [c for c in C.BLOCK_B_LOCATION if c != 'dist_to_loop_ft']
    + C.DEMOGRAPHICS
)
prof = clean.profile_columns(scoped_valid, candidate)

(C.OUTPUTS / 'tables').mkdir(parents=True, exist_ok=True)
prof.to_csv(C.OUTPUTS / 'tables' / 'step0a_profile.csv', index=False)
prof

## 4. Build the analytic sample + engineered feature
`build_analytic_sample` runs the canonical pipeline (scope -> validity -> log target). 
Then we add `dist_to_loop_ft` (our monocentric differentiator).

In [ ]:
print('final analytic sample:', sample.shape)

## 5. Distributions
### 5.1 Target — why we log

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(sample[C.TARGET_RAW].clip(upper=sample[C.TARGET_RAW].quantile(0.99)), bins=60)
axes[0].set_title('sale_price (99th-pct clipped)')
axes[0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
axes[1].hist(sample[C.TARGET], bins=60)
axes[1].set_title('log_sale_price')
plt.tight_layout(); plt.show()

### 5.2 Key predictors

In [ ]:
numeric = [
    'char_bldg_sf', 'char_land_sf', 'char_yrblt', 'char_beds', 'char_fbath',
    'dist_to_loop_ft', 'prox_lake_michigan_dist_ft', 'prox_nearest_cta_stop_dist_ft',
    'acs5_median_income_household_past_year',
]
model_df[numeric].hist(bins=40, figsize=(13, 9))
plt.tight_layout(); plt.show()

## 6. Outlier / sanity analysis
Two things: (a) a price-per-sqft sanity pass, (b) impossible-value checks. 
Remember we *retained* price-extreme sales on purpose — this is about spotting **data errors**, 
not trimming genuine tails.

In [ ]:
model_df['price_per_sqft'] = model_df[C.TARGET_RAW] / model_df['char_bldg_sf'].replace(0, np.nan)
model_df['price_per_sqft'].describe(percentiles=[.01, .05, .5, .95, .99])

In [ ]:
checks = {
    'sale_price <= 10k':   int((pd.to_numeric(model_df[C.TARGET_RAW]) <= 10_000).sum()),
    'char_bldg_sf <= 0':   int((model_df['char_bldg_sf'] <= 0).sum()),
    'char_beds == 0':      int((model_df['char_beds'] == 0).sum()),
    'char_yrblt < 1850':   int((model_df['char_yrblt'] < 1850).sum()),
    'price_per_sqft > 2000': int((model_df['price_per_sqft'] > 2000).sum()),
}
pd.Series(checks, name='n_rows')

### 6.1 Price level by sale year
Motivates the **sale-year fixed effect**: 2022-24 rates moved, so price *levels* drift even within the window.

In [ ]:
sample.boxplot(column=C.TARGET, by=C.YEAR_COL, figsize=(7, 4))
plt.title('log_sale_price by sale year'); plt.suptitle(''); plt.show()

## Next
1. **Calibrate** `config.NON_ARMS_LENGTH_REASONS` from the section-2 output.
2. Decide **school-rating** handling (~25% missing): impute vs complete-case.
3. Lock any sanity drops from section 6, then move to the modeling notebook (`11_inference_hedonic`).